In [ ]:
from pathlib import Path
import pandas as pd
import shutil

# ============================================================
# 1. RUTAS
# ============================================================

REPO = Path.home() / "sylva-ai-fire-smoke"

DATASET = REPO / "dataset_versions" / "dvr_stage2"
IMAGES = DATASET / "images"
LABELS = DATASET / "labels"

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)

# Manifiestos originales utilizados para construir la versión 0.2
SRC_BASE = (
    OUTPUT / "etiquetado_dvr" / "manifest_para_etiquetado.csv"
)

SRC_HARD = (
    OUTPUT / "etiquetado_dvr"
    / "hard_negatives_cloud"
    / "hard_negatives_manifest.csv"
)

MANIFEST_DESTINO = DATASET / "manifest.csv"

for ruta in [IMAGES, LABELS, SRC_BASE, SRC_HARD]:
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")


# ============================================================
# 2. NORMALIZAR LOS 134 REGISTROS INICIALES
# ============================================================

base = pd.read_csv(SRC_BASE)

columnas_requeridas = {
    "archivo",
    "tipo",
    "decision_previa",
    "requiere_cajas",
    "video_origen",
    "segundo_aproximado",
}

faltantes = columnas_requeridas - set(base.columns)

if faltantes:
    raise ValueError(
        f"Faltan columnas en el manifiesto original: {faltantes}"
    )


def nombre_archivo(valor):
    if pd.isna(valor) or not str(valor).strip():
        return ""
    return Path(str(valor)).name


manifest_base = pd.DataFrame({
    "filename": base["archivo"].map(nombre_archivo),
    "group": base["tipo"],
    "initial_review": base["decision_previa"],
    "requires_bounding_boxes": base["requiere_cajas"],
    "source_video": base["video_origen"].map(nombre_archivo),
    "timestamp_seconds": base["segundo_aproximado"],
})


# ============================================================
# 3. NORMALIZAR LOS 30 NEGATIVOS DE NUBES
# ============================================================

hard = pd.read_csv(SRC_HARD)

columnas_hard = {"archivo", "video_origen", "segundo_aproximado"}
faltantes_hard = columnas_hard - set(hard.columns)

if faltantes_hard:
    raise ValueError(
        f"Faltan columnas en el manifiesto de nubes: {faltantes_hard}"
    )

manifest_hard = pd.DataFrame({
    "filename": hard["archivo"].map(nombre_archivo),
    "group": "hard_negative_cloud",
    "initial_review": "background_cloud_confirmed",
    "requires_bounding_boxes": False,
    "source_video": hard["video_origen"].map(nombre_archivo),
    "timestamp_seconds": hard["segundo_aproximado"],
})


# ============================================================
# 4. UNIR LOS MANIFIESTOS CORRECTAMENTE
# ============================================================

manifest_correcto = pd.concat(
    [manifest_base, manifest_hard],
    ignore_index=True
)

# Normalizar nombres vacíos y comprobar duplicados.
manifest_correcto["filename"] = (
    manifest_correcto["filename"]
    .fillna("")
    .astype(str)
    .str.strip()
)

if (manifest_correcto["filename"] == "").any():
    raise ValueError(
        "Hay registros sin nombre de archivo. No se guardó el manifiesto."
    )

duplicados = manifest_correcto[
    manifest_correcto["filename"].duplicated(keep=False)
]

if not duplicados.empty:
    raise ValueError(
        "Se encontraron nombres duplicados:\n"
        + duplicados["filename"].to_string(index=False)
    )


# ============================================================
# 5. VERIFICAR QUE EXISTAN LAS 164 IMÁGENES Y ETIQUETAS
# ============================================================

sin_imagen = []
sin_etiqueta = []

for filename in manifest_correcto["filename"]:

    ruta_img = IMAGES / filename
    ruta_txt = LABELS / f"{Path(filename).stem}.txt"

    if not ruta_img.is_file():
        sin_imagen.append(filename)

    if not ruta_txt.is_file():
        sin_etiqueta.append(filename)

print("========== VALIDACIÓN PREVIA ==========")
print("Registros reconstruidos:", len(manifest_correcto))
print("Imágenes faltantes:", len(sin_imagen))
print("Etiquetas faltantes:", len(sin_etiqueta))

if sin_imagen or sin_etiqueta:
    print("Primeras imágenes faltantes:", sin_imagen[:10])
    print("Primeras etiquetas faltantes:", sin_etiqueta[:10])
    raise FileNotFoundError(
        "Hay archivos faltantes. No se modificó manifest.csv."
    )

if len(manifest_correcto) != 164:
    raise ValueError(
        f"Se esperaban 164 registros, pero hay "
        f"{len(manifest_correcto)}. No se modificó el manifiesto."
    )


# ============================================================
# 6. RESPALDAR Y GUARDAR EL MANIFIESTO REPARADO
# ============================================================

respaldo = (
    Path.home()
    / "sylva_revision"
    / "manifest_anterior_con_columnas_mezcladas.csv"
)

respaldo.parent.mkdir(parents=True, exist_ok=True)

if MANIFEST_DESTINO.exists() and not respaldo.exists():
    shutil.copy2(MANIFEST_DESTINO, respaldo)
    print("Respaldo del manifiesto anterior:", respaldo)

manifest_correcto.to_csv(
    MANIFEST_DESTINO,
    index=False,
    encoding="utf-8-sig"
)

print("\n========== MANIFIESTO REPARADO ==========")
print("Total de registros:", len(manifest_correcto))
print("Imágenes positivas:", int(
    manifest_correcto["group"].isin([
        "fotograma_positivo",
        "imagen_positiva_original"
    ]).sum()
))
print("Fondos y negativos difíciles:", int(
    manifest_correcto["group"].isin([
        "fondo_verificado",
        "hard_negative_cloud"
    ]).sum()
))
print("\nDistribución por grupo:")
print(manifest_correcto["group"].value_counts().to_string())
print("\nGuardado en:", MANIFEST_DESTINO)

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
import random
import re
import math
import shutil

# ============================================================
# 1. RUTAS
# ============================================================

REPO = Path.home() / "sylva-ai-fire-smoke"

DATASET_REPO = REPO / "dataset_versions" / "dvr_stage2"
IMAGENES_REPO = DATASET_REPO / "images"
LABELS_REPO = DATASET_REPO / "labels"
MANIFEST_REPO = DATASET_REPO / "manifest.csv"

OUTPUT = Path(
    "/mnt/d/PROYECTOS/INCENDIOS/DATA_PROCESADA_SYLVA"
)
MANIFEST_NEGATIVOS = OUTPUT / "manifest_frames.csv"

BASE = Path.home() / "IA Vision"
WORK = BASE / "data" / "sylva-dvr-v03-pilot"

# Crear una carpeta de trabajo independiente.
# No modificar el dataset v0.2 ni las grabaciones originales.
if WORK.exists():
    raise FileExistsError(
        f"Ya existe la carpeta de trabajo:\n{WORK}\n"
        "No se modificó ni eliminó. Comprueba su contenido "
        "antes de decidir si debe reutilizarse."
    )

for ruta in [
    IMAGENES_REPO,
    LABELS_REPO,
    MANIFEST_REPO,
    MANIFEST_NEGATIVOS,
]:
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")

manifest_base = pd.read_csv(MANIFEST_REPO)
manifest_neg = pd.read_csv(MANIFEST_NEGATIVOS)

# ============================================================
# 2. SELECCIONAR CUATRO NEGATIVOS POR VÍDEO
# ============================================================

negativos = manifest_neg[
    manifest_neg["grupo_candidato"].astype(str).str.lower()
    == "negativo"
].copy()

negativos["archivo"] = negativos["archivo"].astype(str)
negativos["filename"] = negativos["archivo"].apply(
    lambda x: Path(x).name
)

# Excluir cualquier imagen que ya esté en v0.2.
nombres_existentes = set(
    manifest_base["filename"].astype(str)
)

negativos = negativos[
    ~negativos["filename"].isin(nombres_existentes)
].copy()

# Conservar únicamente archivos existentes.
negativos = negativos[
    negativos["archivo"].map(
        lambda x: Path(x).is_file()
    )
].copy()

MUESTRAS_POR_VIDEO = 4
SEED = 42

muestras = []

for i, (video, grupo) in enumerate(
    negativos.groupby("video_origen", sort=True)
):
    cantidad = min(MUESTRAS_POR_VIDEO, len(grupo))

    seleccion = grupo.sample(
        n=cantidad,
        random_state=SEED + i
    )

    muestras.append(seleccion)

if not muestras:
    raise ValueError(
        "No se encontraron negativos para seleccionar."
    )

df_muestras = pd.concat(
    muestras,
    ignore_index=True
)

print("========== MUESTREO NEGATIVO ==========")
print("Vídeos de origen:", df_muestras["video_origen"].nunique())
print("Fotogramas adicionales:", len(df_muestras))

# ============================================================
# 3. PREPARAR REGISTROS Y AGRUPAR POR VÍDEO
# ============================================================

registros = []

for _, fila in manifest_base.iterrows():

    filename = str(fila["filename"])
    grupo = str(fila["group"])

    source_video = fila.get("source_video", "")
    if pd.isna(source_video):
        source_video = ""
    source_video = str(source_video).strip()

    src_img = IMAGENES_REPO / filename
    src_label = LABELS_REPO / f"{Path(filename).stem}.txt"

    if not src_img.is_file():
        raise FileNotFoundError(f"Falta la imagen: {src_img}")

    if not src_label.is_file():
        raise FileNotFoundError(f"Falta la etiqueta: {src_label}")

    camera = ""

    if grupo == "hard_negative_cloud":
        match = re.match(
            r"(CAMARA[_ ]\d+)_",
            filename,
            flags=re.IGNORECASE
        )

        if not match:
            raise ValueError(
                f"No se pudo identificar la cámara en: {filename}"
            )

        camera = match.group(1).replace("_", " ").upper()
        group_key = (
            f"NEG::{camera.casefold()}::{source_video.casefold()}"
        )

    elif source_video and source_video.lower() != "nan":
        group_key = f"POS::{source_video.casefold()}"

    else:
        # Las siete imágenes originales se dejan en entrenamiento.
        group_key = f"STILL::{filename.casefold()}"

    registros.append({
        "filename": filename,
        "source_image": str(src_img),
        "source_label": str(src_label),
        "dataset_group": grupo,
        "source_video": source_video,
        "camera": camera,
        "split_group": group_key,
        "is_new_negative": False,
    })

# Agregar los nuevos fotogramas negativos seleccionados.
for _, fila in df_muestras.iterrows():

    src_img = Path(str(fila["archivo"]))
    filename = src_img.name

    video_path = Path(str(fila["video_origen"]))
    video_name = video_path.name
    camera = video_path.parent.name

    group_key = (
        f"NEG::{camera.casefold()}::{video_name.casefold()}"
    )

    registros.append({
        "filename": filename,
        "source_image": str(src_img),
        "source_label": "",
        "dataset_group": "negative_camera_sample",
        "source_video": video_name,
        "camera": camera,
        "split_group": group_key,
        "is_new_negative": True,
    })

df_dataset = pd.DataFrame(registros)

duplicados = df_dataset[
    df_dataset["filename"].duplicated(keep=False)
]

if not duplicados.empty:
    raise ValueError(
        "Hay nombres de archivo duplicados. "
        "No se creó el dataset de trabajo:\n"
        + duplicados["filename"].to_string(index=False)
    )

# ============================================================
# 4. SEPARAR VALIDACIÓN POR VÍDEO DE ORIGEN
# ============================================================

# Reservar una grabación de humo y otra de fuego para validación.
# Las otras grabaciones positivas quedan en entrenamiento.
grupos_validacion = {
    "POS::cinco.mp4",
    "POS::dos.mp4",
}

grupos_disponibles = set(df_dataset["split_group"])

faltantes = grupos_validacion - grupos_disponibles

if faltantes:
    raise ValueError(
        "No se encontraron los grupos positivos esperados: "
        + str(faltantes)
    )

# Separar aproximadamente el 20% de los vídeos negativos por cámara.
# Todos los fotogramas de un vídeo quedan en el mismo split.
rng = random.Random(SEED)
grupos_por_camara = defaultdict(set)

for _, fila in df_dataset.iterrows():
    if str(fila["split_group"]).startswith("NEG::"):
        grupos_por_camara[str(fila["camera"])].add(
            fila["split_group"]
        )

for camera, grupos in sorted(grupos_por_camara.items()):

    grupos = sorted(grupos)
    if len(grupos) < 2:
        continue

    n_val = max(1, math.ceil(len(grupos) * 0.20))
    n_val = min(n_val, len(grupos) - 1)

    grupos_validacion.update(
        rng.sample(grupos, n_val)
    )

df_dataset["split"] = df_dataset["split_group"].apply(
    lambda grupo: (
        "val" if grupo in grupos_validacion else "train"
    )
)

# Ningún grupo debe aparecer a ambos lados.
grupos_train = set(
    df_dataset.loc[df_dataset["split"] == "train", "split_group"]
)
grupos_val = set(
    df_dataset.loc[df_dataset["split"] == "val", "split_group"]
)

assert not (grupos_train & grupos_val), (
    "Error: hay vídeos repartidos entre entrenamiento y validación."
)

# ============================================================
# 5. CREAR CARPETAS DE TRABAJO
# ============================================================

for split in ["train", "val"]:
    (WORK / "images" / split).mkdir(
        parents=True,
        exist_ok=True
    )
    (WORK / "labels" / split).mkdir(
        parents=True,
        exist_ok=True
    )

# Una carpeta nueva se crea por esta celda.
# Se usa symlink para las imágenes y copia para los TXT.
for _, fila in df_dataset.iterrows():

    split = fila["split"]
    filename = fila["filename"]

    src_img = Path(fila["source_image"])
    dst_img = WORK / "images" / split / filename
    dst_label = (
        WORK / "labels" / split / f"{Path(filename).stem}.txt"
    )

    if not src_img.is_file():
        raise FileNotFoundError(f"No existe la imagen: {src_img}")

    try:
        dst_img.symlink_to(src_img.resolve())
    except OSError:
        # Alternativa si el sistema no admite enlaces simbólicos.
        shutil.copy2(src_img, dst_img)

    if fila["is_new_negative"]:
        # Fondo conocido: sin cajas.
        dst_label.write_text("", encoding="utf-8")
    else:
        src_label = Path(fila["source_label"])
        shutil.copy2(src_label, dst_label)

# Guardar manifiesto de esta división.
df_dataset.to_csv(
    WORK / "manifest_split.csv",
    index=False,
    encoding="utf-8-sig"
)

# Archivo de configuración de Ultralytics.
ruta_yaml = WORK / "data.yaml"

ruta_yaml.write_text(
    f'path: "{WORK.as_posix()}"\n'
    "train: images/train\n"
    "val: images/val\n"
    "nc: 2\n"
    "names:\n"
    "  0: smoke\n"
    "  1: fire\n",
    encoding="utf-8"
)

# ============================================================
# 6. AUDITORÍA DEL SPLIT
# ============================================================

print("\n========== DATASET PILOTO V0.3 ==========")
print("Total de imágenes:", len(df_dataset))
print("Entrenamiento:", int((df_dataset["split"] == "train").sum()))
print("Validación:", int((df_dataset["split"] == "val").sum()))
print("Vídeos negativos seleccionados:", len(df_muestras.groupby("video_origen")))
print("Grupos separados entre train y val:", len(grupos_train & grupos_val))

for split in ["train", "val"]:

    filas_split = df_dataset[df_dataset["split"] == split]
    conteos = Counter()

    for _, fila in filas_split.iterrows():
        label_destino = (
            WORK / "labels" / split
            / f"{Path(fila['filename']).stem}.txt"
        )

        for linea in label_destino.read_text(
            encoding="utf-8"
        ).splitlines():

            partes = linea.split()
            if len(partes) == 5:
                try:
                    clase = int(float(partes[0]))
                    if clase == 0:
                        conteos["smoke"] += 1
                    elif clase == 1:
                        conteos["fire"] += 1
                except ValueError:
                    pass

    print(
        f"\n{split.upper()}: "
        f"{len(filas_split)} imágenes | "
        f"{conteos['smoke']} cajas de humo | "
        f"{conteos['fire']} cajas de fuego"
    )

print("\nYAML creado:", ruta_yaml)
print("Manifiesto:", WORK / "manifest_split.csv")

Activar el registro de MLflow

In [ ]:
import os
import mlflow
from ultralytics import settings

# Conectar el notebook al servidor local
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"

# Agrupar los experimentos de Sylva AI
os.environ["MLFLOW_EXPERIMENT_NAME"] = "SylvaAI_DVR"

# Habilitar integración de Ultralytics
settings.update({"mlflow": True})

print("URI de seguimiento:", mlflow.get_tracking_uri())
print("MLflow habilitado:", settings.get("mlflow"))
print("Experimento:", os.environ["MLFLOW_EXPERIMENT_NAME"])

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import torch
import pandas as pd

# ============================================================
# 1. VERIFICAR RUTAS Y GPU
# ============================================================

BASE = Path.home() / "IA Vision"

PESO_INICIAL = (
    BASE / "runs" / "detect"
    / "yolov8s-odiug77-split"
    / "weights" / "best.pt"
)

DATA_YAML = (
    BASE / "data" / "sylva-dvr-v03-pilot" / "data.yaml"
)

assert PESO_INICIAL.exists(), (
    f"No se encontraron los pesos iniciales: {PESO_INICIAL}"
)
assert DATA_YAML.exists(), (
    f"No se encontró el YAML: {DATA_YAML}"
)

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. CARGAR EL MEJOR YOLOV8S
# ============================================================

modelo_dvr = YOLO(str(PESO_INICIAL))

print("Pesos iniciales:", PESO_INICIAL)
print("Clases:", modelo_dvr.names)

assert modelo_dvr.names == {0: "smoke", 1: "fire"}, (
    "El orden de clases del modelo no coincide con el dataset."
)

# ============================================================
# 3. ENTRENAMIENTO PILOTO
# ============================================================

resultados_train_dvr = modelo_dvr.train(
    data=str(DATA_YAML),

    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=4,

    patience=7,
    seed=42,
    deterministic=True,

    # Ajustes conservadores para un dataset pequeño.
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=2,
    close_mosaic=5,

    cache=False,
    plots=True,
    save=True,
    verbose=True,

    project=str(BASE / "runs" / "detect"),
    name="yolov8s-sylva-dvr-v03-pilot",
)

ruta_best_dvr = Path(modelo_dvr.trainer.best)

print("\n========== ENTRENAMIENTO TERMINADO ==========")
print("Mejores pesos:", ruta_best_dvr)
print("¿Existe best.pt?:", ruta_best_dvr.exists())
print("Resultados:", modelo_dvr.trainer.save_dir)

if not ruta_best_dvr.exists():
    raise FileNotFoundError(
        "No se encontró best.pt después del entrenamiento."
    )

# ============================================================
# 4. COMPARAR ANTES Y DESPUÉS EN EL MISMO SPLIT
# ============================================================

modelos_comparacion = [
    ("YOLOv8s base", PESO_INICIAL),
    ("YOLOv8s ajustado a DVR", ruta_best_dvr),
]

filas = []

for nombre, ruta_pesos in modelos_comparacion:

    print(f"\nEvaluando: {nombre}")

    modelo_eval = YOLO(str(ruta_pesos))

    r = modelo_eval.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=640,
        batch=4,
        device=0,
        plots=True,
        verbose=False,
        project=str(BASE / "runs" / "detect"),
        name=f"eval-{ruta_pesos.parent.parent.parent.name}",
        exist_ok=True
    )

    filas.append({
        "Modelo": nombre,
        "Precision": float(r.box.mp),
        "Recall": float(r.box.mr),
        "mAP50": float(r.box.map50),
        "mAP50-95": float(r.box.map),
        "Recall_smoke": float(r.box.r[0]),
        "Recall_fire": float(r.box.r[1]),
        "Inferencia_ms": float(r.speed["inference"]),
    })

comparacion_dvr = pd.DataFrame(filas)

print("\n========== COMPARACIÓN EN VALIDACIÓN DVR ==========")
print(comparacion_dvr.round(4).to_string(index=False))